[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1aLupgl-PLKbinibH4wEe1vkHk7JbdKrL/view?usp=sharing)

# Prompt Evaluation – With RAG Contexts

This notebook demonstrates how to compare prompts when your pipeline uses retrieved context. Samples include `contexts`; Floeval generates responses from question + context for each prompt, then scores with faithfulness.

**Objectives**
- Install Floeval and configure credentials
- Provide paths to prompts YAML and a partial dataset JSON with `contexts`
- Run prompt evaluation with RAG metrics (answer_relevancy, faithfulness)
- Inspect results by prompt

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
%pip install floeval

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) - set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass

# LLM and API configuration (OpenAI)
OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("Enter your API key: ")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

Import Floeval evaluation classes and the provider config schema.

In [ ]:
from pathlib import Path

from floeval import DatasetLoader, Evaluation
from floeval.config.schemas.io.llm import OpenAIProviderConfig


## 4. Configure the LLM

Build an OpenAI-compatible provider config used for response generation and metric scoring.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)

## 5. Load Prompts YAML and Dataset

**Prompts YAML:**

```yaml
prompts:
  "1":
    template: "System instruction text"
```

**Dataset** (with `contexts` and `prompt_ids` per sample): JSON with `samples`, or JSONL one object per line.

```json
{
  "samples": [
    { "user_input": "...", "contexts": ["..."], "prompt_ids": ["1", "2"] }
  ]
}
```

**Example files**  
<a href="../datasets/prompt_evaluation/sample_prompts_with_contexts.yaml" download="sample_prompts_with_contexts.yaml">sample_prompts_with_contexts.yaml</a><br>
<a href="../datasets/prompt_evaluation/partial_dataset_squad_top50_prompt_context.jsonl" download="partial_dataset_squad_top50_prompt_context.jsonl">partial_dataset_squad_top50_prompt_context.jsonl</a>

Provide prompts and dataset paths (`.yaml` + `.jsonl`/`.json`), then load the context-aware dataset with `DatasetLoader`.

In [ ]:
try:
    from google.colab import files

    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload prompts YAML file:")
    uploaded_prompts = files.upload()
    if not uploaded_prompts:
        raise RuntimeError("No prompts file uploaded.")
    prompts_path = Path(next(iter(uploaded_prompts.keys())))
    print("Upload dataset .jsonl/.json file:")
    uploaded_ds = files.upload()
    if not uploaded_ds:
        raise RuntimeError("No dataset file uploaded.")
    dataset_path = Path(next(iter(uploaded_ds.keys())))
else:
    prompts_path = (
        Path(input("Enter path to prompts YAML file: ").strip().strip('"')).expanduser().resolve()
    )
    if not prompts_path.exists():
        raise FileNotFoundError(f"File not found: {prompts_path}")
    print("✅ Prompts file found.")
    dataset_path = (
        Path(input("Enter path to dataset .jsonl/.json file: ").strip().strip('"'))
        .expanduser()
        .resolve()
    )
    if not dataset_path.exists():
        raise FileNotFoundError(f"File not found: {dataset_path}")
    print("✅ Dataset file found.")


### Resolve Prompts and Dataset Paths

Provide `prompts_path` and `dataset_path` via uploads in Colab or local path input in Jupyter (dataset: `.jsonl` or `.json`).


In [ ]:
dataset = DatasetLoader.from_file(dataset_path, partial_dataset=True)
print("Prompts:", prompts_path)
print("Dataset:", dataset_path, f"— {len(dataset.samples)} samples")


### Load Dataset with Contexts

Load the partial prompt dataset (including `contexts`) with `DatasetLoader.from_file(..., partial_dataset=True)`.


## 6. Create and Run the Evaluation

Use `answer_relevancy` and `faithfulness` to compare how each prompt performs with retrieved contexts.

In [ ]:
evaluation = Evaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=["answer_relevancy", "faithfulness"],
    default_provider="ragas",
    dataset_generator_model=OPENAI_CHAT_MODEL,
    prompts_file=str(prompts_path),
)


### Build Evaluation Object

Configure `Evaluation` with prompt variants, retrieved contexts, and RAG metrics.


In [ ]:
results = await evaluation.arun()
print("Aggregate scores:", results.aggregate_scores)


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


### Run Evaluation (Async)

Execute `await evaluation.arun()` to generate responses and compute prompt-level scores.


## 7. Inspect Results by Prompt

Each result includes `prompt_id` and generated `llm_response`, so you can compare metric behavior across prompt variants.

### Inspect per-sample results

Iterates `results.sample_results` to print each question snippet and metric scores.


In [ ]:
for sr in results.sample_results:
    pid = sr.get("prompt_id", "unknown")
    metrics = sr.get("metrics", {})
    print(f"Prompt: {pid}")
    for k, v in metrics.items():
        print(f"  {k}: {v.get('score')}")

## Summary

This notebook demonstrated how to compare prompts when the pipeline uses retrieved context.

The key components included:

1. **Prompts File**: Paths pointed to a YAML file with context-aware instructions.
2. **Dataset with Contexts**: A partial dataset was loaded from JSON with `user_input`, `contexts`, and `prompt_ids`.
3. **RAG Metrics**: The `answer_relevancy` and `faithfulness` metrics were used to evaluate grounding quality.
4. **Results by Prompt**: Results were inspected by `prompt_id` to compare grounding quality across instructions.

This example showcases prompt evaluation with RAG contexts and faithfulness scoring.